In [3]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())

Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [4]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))

Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


Task 1 — Multi-series line with highlight

In [5]:

asia_df = df[df['Region'] == 'Asia']

highlight_country = 'China'

fig = go.Figure()

for country in asia_df['Country'].unique():
    country_df = asia_df[asia_df['Country'] == country]

    if country == highlight_country:
        fig.add_trace(go.Scatter(
            x=country_df['Year'],
            y=country_df['CO2_Mt'],
            mode='lines',
            line=dict(color='red', width=3),
            showlegend=False
        ))

        # label at end of line
        fig.add_annotation(
            x=country_df['Year'].max(),
            y=country_df[country_df['Year'] == country_df['Year'].max()]['CO2_Mt'].values[0],
            text=country,
            showarrow=False,
            font=dict(color='red', family='Arial')
        )

    else:
        fig.add_trace(go.Scatter(
            x=country_df['Year'],
            y=country_df['CO2_Mt'],
            mode='lines',
            line=dict(color='#DDDDDD', width=1),
            showlegend=False
        ))

fig.update_layout(
    title="China’s CO₂ emissions rose dramatically, outpacing all other Asian countries since 2000",
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial'),
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False),
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()

Task 2 — Slopegraph: regional averages

In [6]:

regional_df = df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()
slope_df = regional_df[regional_df['Year'].isin([2000, 2022])]

fig = go.Figure()

for region in slope_df['Region'].unique():
    region_data = slope_df[slope_df['Region'] == region].sort_values('Year')

    y0 = region_data.iloc[0]['CO2_Mt']
    y1 = region_data.iloc[1]['CO2_Mt']

    color = 'red' if y1 > y0 else 'green'

    fig.add_trace(go.Scatter(
        x=[2000, 2022],
        y=[y0, y1],
        mode='lines+markers',
        line=dict(color=color, width=2),
        showlegend=False
    ))

    # label for 2000
    fig.add_annotation(
        x=2000,
        y=y0,
        text=f"{region}: {round(y0,1)}",
        showarrow=False,
        xanchor='right',
        font=dict(family='Arial')
    )

    # label for 2022
    fig.add_annotation(
        x=2022,
        y=y1,
        text=f"{region}: {round(y1,1)}",
        showarrow=False,
        xanchor='left',
        font=dict(family='Arial')
    )

fig.update_layout(
    title="CO₂ emissions increased across most regions since 2000, with Asia showing the largest rise",
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial'),
    xaxis=dict(showgrid=False, tickvals=[2000, 2022]),
    yaxis=dict(showgrid=False, showticklabels=False),
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()